# CCTV · Phát hiện bất thường theo từng vật thể, đa nhiệm tự giám sát (AUROC) - KHÔNG dùng trọng số pretrained

Bất thường trong bài nằm ở **từng vật thể** (đi sai làn, xe đạp trên vỉa hè, băng qua đường sai chỗ, xe đỗ trái phép),
nên mô hình học "vật thể bình thường trông thế nào, ở đâu, di chuyển ra sao" chỉ từ các video train (toàn cảnh bình thường).

| Giai đoạn | Nội dung |
|---|---|
| **0. Tách vật thể** (cv2, không học) | so với nền trung vị để có mặt nạ, connected components để có hộp, optical flow Farneback; cắt "ống vật thể" 5 khung (t-2..t+2) + flow, kích thước 32×32, **che nền** (chỉ giữ mặt nạ vật thể, để mạng không học thuộc nền) |
| **1. Mạng đa nhiệm** (train từ đầu) | encoder ResNet nhỏ chung, bộ nhớ MemAE, 5 nhánh: **H1** tái tạo khung t (qua bộ nhớ), **H2** mũi tên thời gian (ống xuôi hay đảo), **H3** nhảy cóc (khung liên tiếp hay cách 1), **H4** vị trí (đoán ô lưới 6×10 nơi vật thể đứng), **H5** tái tạo optical flow |
| **2. Từ vật thể ra khung** | chuẩn hóa từng loại lỗi theo vật thể của video train để riêng, điểm khung = max theo vật thể, kết hợp nhánh toàn cảnh (`pix`), làm mượt theo thời gian |

Val chỉ có 7 video và không đại diện tốt cho test, nên notebook **ghi sẵn nhiều biến thể** để chọn bằng LB public.
Thư viện: `torch`, `opencv-python`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`.

In [ ]:
import os, glob, time, math, random
import torch          # import trước pandas để tránh lỗi DLL trên Windows
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import cv2
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

DATA = '.'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
S = 32                  # kích thước ống vật thể
MAXOBJ = 10             # tối đa số vật thể mỗi khung (lớn nhất trước)
GR, GC = 6, 10          # lưới vị trí 6 x 10 ô
T_BG, MIN_AREA = 25, 40
BUDGET = 20             # phút train mạng
BS = 256
T0 = time.time()
_last = [0.0]
open('run_log.txt', 'w').close()


def log(*a, every=0):
    """In + ghi run_log.txt. every > 0: chỉ ghi nếu đã qua `every` giây kể từ lần ghi trước (để báo tiến độ ~3 phút/lần)."""
    if every and time.time() - _last[0] < every:
        return
    _last[0] = time.time()
    msg = f'[{(time.time() - T0) / 60:5.1f} phút] ' + ' '.join(map(str, a)); print(msg)
    with open('run_log.txt', 'a', encoding='utf-8') as f: f.write(msg + '\n')


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


splits = {s: {v: sorted(glob.glob(f'{DATA}/{s}/{v}/*.jpg')) for v in sorted(os.listdir(f'{DATA}/{s}'))}
          for s in ['train', 'validation', 'test']}
fid = lambda p: f'{os.path.basename(os.path.dirname(p))}_{os.path.basename(p)[:-4]}'
lab = pd.read_csv(f'{DATA}/validation_labels.csv').set_index('id')['label']
read = lambda p: cv2.imread(p, cv2.IMREAD_GRAYSCALE)
log(torch.__version__, DEV, {s: sum(map(len, d.values())) for s, d in splits.items()})

## 1. Nền trung vị + thống kê từng điểm ảnh từ train

In [ ]:
tr_paths = [p for v in splits['train'].values() for p in v]
stack = np.stack([read(p) for p in tr_paths[::5]])
BG = np.median(stack, 0).astype(np.uint8)
BG_BLUR = cv2.GaussianBlur(BG, (5, 5), 0)
small = np.stack([cv2.resize(f, (80, 45), interpolation=cv2.INTER_AREA) for f in stack]).astype(np.float32)
BG_S = np.median(small, 0)
MAD_S = np.median(np.abs(small - BG_S), 0) + 1.0
del stack, small
log('xong nền trung vị', BG.shape)

## 2. Giai đoạn 0 · Tách vật thể và cắt "ống vật thể"

In [ ]:
K3, K7 = np.ones((3, 3), np.uint8), np.ones((7, 7), np.uint8)


def patch(img, cx, cy, side):
    return cv2.resize(cv2.getRectSubPix(img, (side, side), (float(cx), float(cy))), (S, S), interpolation=cv2.INTER_AREA)


def fg_mask(f):
    m = (cv2.absdiff(cv2.GaussianBlur(f, (5, 5), 0), BG_BLUR) > T_BG).astype(np.uint8)
    return cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, K3), cv2.MORPH_CLOSE, K7)


def process_video(paths):
    fr = [read(p) for p in paths]; n = len(fr); H, W = fr[0].shape
    ms = [fg_mask(f) for f in fr]
    mk = [cv2.dilate(m, K7) * 255 for m in ms]          # mặt nạ nới rộng: giữ trọn vật thể, bỏ nền
    out = {'tube': [], 'flow': [], 'cell': [], 'box': [], 'fidx': []}
    glob_feats = []
    for t in range(n):
        f, m = fr[t], ms[t]
        nl, _, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        comps = sorted([i for i in range(1, nl) if st[i, cv2.CC_STAT_AREA] >= MIN_AREA],
                       key=lambda i: -st[i, cv2.CC_STAT_AREA])[:MAXOBJ]
        sm = cv2.resize(f, (80, 45), interpolation=cv2.INTER_AREA).astype(np.float32)   # nhánh toàn cảnh
        dd = np.abs(sm - BG_S)
        glob_feats.append((np.percentile(dd, 99.5) + dd.mean(), np.sort((dd / MAD_S).ravel())[-72:].mean(), len(comps)))
        if not comps:
            continue
        flow = cv2.calcOpticalFlowFarneback(fr[max(t - 1, 0)], f, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        fx, fy = np.ascontiguousarray(flow[..., 0]), np.ascontiguousarray(flow[..., 1])
        for i in comps:
            x, y, w, h = st[i, :4]
            side = int(max(w, h) * 1.3) + 8
            cx, cy = x + w / 2, y + h / 2
            tube = []
            for k in (-2, -1, 0, 1, 2):
                tt = min(max(t + k, 0), n - 1)
                tube.append(patch(fr[tt], cx, cy, side) * (patch(mk[tt], cx, cy, side) > 127))   # che nền
            out['tube'].append(np.stack(tube))
            keep = patch(mk[t], cx, cy, side) > 127
            fl = np.stack([patch(fx, cx, cy, side), patch(fy, cx, cy, side)]) * (S / side) * keep  # flow chỉ trên vật thể
            out['flow'].append(np.clip(np.round(fl * 16), -127, 127).astype(np.int8))
            out['cell'].append(min(int(cy / H * GR), GR - 1) * GC + min(int(cx / W * GC), GC - 1))
            out['box'].append((cx / W, cy / H, side / W))
            out['fidx'].append(t)
    return out, glob_feats


def run_split(name):
    rows, T_, FL, CELL, BOX, FRAME = [], [], [], [], [], []
    vids, t0, base = list(splits[name].items()), time.time(), 0
    for k, (v, ps) in enumerate(vids):
        out, g = process_video(ps)
        rows += [{'id': fid(p), 'video': v, 'bg': g[t][0], 'pix': g[t][1], 'nobj': g[t][2]} for t, p in enumerate(ps)]
        if out['tube']:
            T_.append(np.stack(out['tube'])); FL.append(np.stack(out['flow'])); CELL.append(np.array(out['cell']))
            BOX.append(np.array(out['box'], np.float32)); FRAME.append(np.array(out['fidx']) + base)
        base += len(ps)
        log(f'  tách vật thể {name}: {k + 1}/{len(vids)} video · còn ~{(time.time() - t0) / (k + 1) * (len(vids) - k - 1) / 60:.1f} phút', every=170)
    df = pd.DataFrame(rows)
    obj = dict(tube=np.concatenate(T_), flow=np.concatenate(FL), cell=np.concatenate(CELL),
               box=np.concatenate(BOX), frame=np.concatenate(FRAME))
    log(f'{name}: {len(df)} khung, {len(obj["cell"]):,} vật thể (TB {len(obj["cell"]) / len(df):.1f}/khung)')
    return df, obj


CACHE = 'cache_obj_masked'          # cache vật thể đã tách để lần sau khỏi tách lại (~5 phút)
os.makedirs(CACHE, exist_ok=True)


def run_split_cached(name):
    f = f'{CACHE}/{name}.npz'
    if os.path.exists(f):
        z = np.load(f)
        obj = {k: z[k] for k in ['tube', 'flow', 'cell', 'box', 'frame']}
        df = pd.read_csv(f'{CACHE}/{name}.csv')
        log(f'{name}: nạp cache {len(obj["cell"]):,} vật thể')
        return df, obj
    df, obj = run_split(name)
    np.savez(f, **obj); df.to_csv(f'{CACHE}/{name}.csv', index=False)
    return df, obj


df_tr, ob_tr = run_split_cached('train')
df_va, ob_va = run_split_cached('validation')
df_te, ob_te = run_split_cached('test')
df_va['y'] = lab.loc[df_va.id].values

Kiểm tra bằng mắt: hộp vật thể trên vài khung val (đỏ = khung bất thường) và phân bố vị trí vật thể trong train.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
pick = list(df_va.index[df_va.y == 1][::150][:3]) + list(df_va.index[df_va.y == 0][::400][:2])
for a, r in zip(ax.ravel(), pick):
    v, sec = df_va.id[r].split('_')
    img = cv2.cvtColor(cv2.imread(f'{DATA}/validation/{v}/{sec}.jpg'), cv2.COLOR_BGR2RGB)
    for cx, cy, sd in ob_va['box'][ob_va['frame'] == r]:
        H, W = img.shape[:2]; s = sd * W / 2
        cv2.rectangle(img, (int(cx * W - s), int(cy * H - s)), (int(cx * W + s), int(cy * H + s)), (255, 0, 0) if df_va.y[r] else (0, 255, 0), 1)
    a.imshow(img); a.set_title(f'{df_va.id[r]} nhãn={df_va.y[r]}'); a.axis('off')
ax.ravel()[-1].imshow(np.bincount(ob_tr['cell'], minlength=GR * GC).reshape(GR, GC), cmap='viridis')
ax.ravel()[-1].set_title('số vật thể train theo ô lưới'); plt.tight_layout(); plt.show()

## 3. Giai đoạn 1 · Mạng đa nhiệm tự giám sát (train từ đầu)
Video train để riêng (`HOLD` video cuối) không dùng để train mà để tính thống kê chuẩn hóa lỗi của vật thể bình thường.

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.sk = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
                   if stride != 1 or cin != cout else nn.Identity())

    def forward(self, x):
        return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x))))) + self.sk(x))


class Net(nn.Module):
    def __init__(self, n_mem=100, d=128):
        super().__init__()
        self.d = d
        self.enc = nn.Sequential(nn.Conv2d(5, 24, 3, 1, 1, bias=False), nn.BatchNorm2d(24), nn.ReLU(),
                                 Block(24, 48, 2), Block(48, 96, 2), Block(96, d, 2),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.mem = nn.Parameter(torch.randn(n_mem, d) / math.sqrt(d))          # bộ nhớ MemAE
        self.fc = nn.Linear(d, d * 16)
        self.dec = nn.Sequential(nn.ConvTranspose2d(d, 96, 4, 2, 1), nn.BatchNorm2d(96), nn.ReLU(),
                                 nn.ConvTranspose2d(96, 48, 4, 2, 1), nn.BatchNorm2d(48), nn.ReLU(),
                                 nn.ConvTranspose2d(48, 24, 4, 2, 1), nn.BatchNorm2d(24), nn.ReLU(),
                                 nn.Conv2d(24, 3, 3, 1, 1))
        self.drop = nn.Dropout(0.3)           # trước các nhánh phân loại
        self.loc = nn.Linear(d, GR * GC)      # H4
        self.arrow = nn.Linear(d, 2)          # H2
        self.skip = nn.Linear(d, 2)           # H3

    def memory(self, z):
        att = F.softmax(z @ self.mem.t(), dim=1)
        lam = 1.0 / self.mem.shape[0]
        att = F.relu(att - lam) * att / (torch.abs(att - lam) + 1e-12)        # hard shrinkage
        att = att / (att.sum(1, keepdim=True) + 1e-12)
        return att @ self.mem, att

    def decode(self, z):
        zm, att = self.memory(z)
        out = self.dec(self.fc(zm).view(-1, self.d, 4, 4))
        return torch.sigmoid(out[:, :1]), torch.tanh(out[:, 1:]), att        # H1 khung t, H5 flow


def jitter(fr):
    """Cùng độ sáng / tương phản cho cả ống (không lật: lật làm đổi chiều chuyển động)."""
    a = torch.empty(fr.shape[0], 1, 1, 1, device=fr.device).uniform_(0.8, 1.2)
    b = torch.empty(fr.shape[0], 1, 1, 1, device=fr.device).uniform_(-0.1, 0.1)
    return (fr * a + b).clamp(0, 1)


def tasks(tu, fl, train):
    fr, fl = tu.float() / 255, fl.float() / 127
    if train:
        fr = jitter(fr)
    z0 = torch.zeros_like(fl)
    main = torch.cat([fr[:, 1:4], fl], 1)                 # khung t-1, t, t+1 + flow
    fwd = torch.cat([fr[:, 1:4], z0], 1)                  # chỉ khung, đúng thứ tự
    rev = torch.cat([fr[:, [3, 2, 1]], z0], 1)            # đảo thời gian
    skp = torch.cat([fr[:, [0, 2, 4]], z0], 1)            # nhảy cóc t-2, t, t+2
    return main, fwd, rev, skp, fr[:, 2:3], fl


HOLD = 2
hold_vids = sorted(splits['train'])[-HOLD:]
is_hold = np.isin(df_tr.video.values[ob_tr['frame']], hold_vids)
TU = torch.from_numpy(ob_tr['tube'][~is_hold]).to(DEV)
FLW = torch.from_numpy(ob_tr['flow'][~is_hold]).to(DEV)
CELL = torch.from_numpy(ob_tr['cell'][~is_hold]).long().to(DEV)
log(f'vật thể train {len(CELL):,} · để riêng {is_hold.sum():,} (video {hold_vids})')


def train_model(budget_min, lr=2e-3, max_epochs=60):
    seed_all()
    model = Net().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    N = len(CELL); spe = N // BS
    sched = None

    def step(idx, update=True):
        main, fwd, rev, skp, tf, tfl = tasks(TU[idx], FLW[idx], train=True)
        B, h = len(idx), len(idx) // 2
        y = torch.cat([torch.ones(h), torch.zeros(B - h)]).long().to(DEV)
        z = model.enc(main)
        pf, pfl, att = model.decode(z)
        ls = {'H1': F.l1_loss(pf, tf), 'H5': F.l1_loss(pfl, tfl), 'H4': F.cross_entropy(model.loc(model.drop(z)), CELL[idx]),
              'H2': F.cross_entropy(model.arrow(model.drop(model.enc(torch.cat([fwd[:h], rev[h:]])))), y),
              'H3': F.cross_entropy(model.skip(model.drop(model.enc(torch.cat([fwd[:h], skp[h:]])))), y),
              'ent': (-att * torch.log(att + 1e-12)).sum(1).mean()}
        loss = ls['H1'] + ls['H5'] + ls['H4'] + 0.5 * ls['H2'] + 0.5 * ls['H3'] + 2e-4 * ls['ent']
        opt.zero_grad(set_to_none=True); loss.backward()
        if update:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); opt.step(); sched.step()
        return {k: v.item() for k, v in ls.items()}

    model.train()
    rnd = lambda: torch.randint(0, N, (BS,), device=DEV)
    for _ in range(2): step(rnd(), update=False)
    if DEV == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(5): step(rnd(), update=False)
    if DEV == 'cuda': torch.cuda.synchronize()
    sec = (time.time() - t0) / 5
    total = int(min(budget_min * 60 / sec, max_epochs * spe))
    log(f'{sec * 1000:.0f} ms/bước · chạy {total} bước ({total / spe:.1f} epoch) ≈ {total * sec / 60:.1f} phút')
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total, pct_start=0.15)
    run, t1 = {}, time.time()
    for it in range(total):
        if it % spe == 0:
            perm = torch.randperm(N, device=DEV)
        ls = step(perm[(it % spe) * BS:(it % spe + 1) * BS])
        assert all(np.isfinite(v) for v in ls.values()), f'loss NaN/inf: {ls}'
        run = {k: run.get(k, v) * 0.98 + v * 0.02 for k, v in ls.items()}
        log(f'  bước {it + 1}/{total} · ' + ' '.join(f'{k} {v:.3f}' for k, v in run.items())
            + f' · còn ~{(time.time() - t1) / (it + 1) * (total - it - 1) / 60:.1f} phút', every=170)
    log('train xong · loss cuối: ' + ' '.join(f'{k} {v:.3f}' for k, v in run.items()))
    return model


model = train_model(BUDGET)
torch.save(model.state_dict(), 'cctv_object_model.pt')
del TU, FLW, CELL; torch.cuda.empty_cache()

## 4. Giai đoạn 2 · Điểm vật thể, điểm khung

In [ ]:
COMPS = ['H1', 'H5', 'H4', 'H2', 'H3']


@torch.no_grad()
def score_objects(obj, bs=1024):
    model.eval(); res = {k: [] for k in COMPS}
    for i in range(0, len(obj['cell']), bs):
        tu = torch.from_numpy(obj['tube'][i:i + bs]).to(DEV); fl = torch.from_numpy(obj['flow'][i:i + bs]).to(DEV)
        cell = torch.from_numpy(obj['cell'][i:i + bs]).long().to(DEV)
        main, fwd, _, _, tf, tfl = tasks(tu, fl, train=False)
        z = model.enc(main); pf, pfl, _ = model.decode(z); zf = model.enc(fwd)
        res['H1'].append((pf - tf).abs().mean((1, 2, 3)).cpu())
        res['H5'].append((pfl - tfl).abs().mean((1, 2, 3)).cpu())
        res['H4'].append(-F.log_softmax(model.loc(z), 1).gather(1, cell[:, None])[:, 0].cpu())
        res['H2'].append(-F.log_softmax(model.arrow(zf), 1)[:, 1].cpu())          # -log p(xuôi)
        res['H3'].append(-F.log_softmax(model.skip(zf), 1)[:, 1].cpu())           # -log p(liên tiếp)
    return pd.DataFrame({k: torch.cat(v).numpy() for k, v in res.items()})


sc_hold = score_objects({k: v[is_hold] for k, v in ob_tr.items()})
MU, SD = sc_hold.mean(), sc_hold.std() + 1e-6
log('điểm vật thể bình thường (video để riêng) - trung bình:', MU.round(3).to_dict())


def frame_table(df, obj):
    z = (score_objects(obj)[COMPS] - MU) / SD
    z['total'] = z[COMPS].sum(1)
    z['frame'] = obj['frame']
    agg = z.groupby('frame').max()
    out = df.copy()
    for c in COMPS + ['total']:
        out[c] = agg[c].reindex(range(len(df))).fillna(agg[c].min()).values     # khung không có vật thể: điểm thấp nhất
    return out


def smooth(df, x, sig=1):
    out = np.asarray(x, float).copy()
    for v in df.video.unique():
        i = np.where(df.video.values == v)[0]
        out[i] = gaussian_filter1d(out[i], sig) if sig else out[i]
    return out


va = frame_table(df_va, ob_va)
te = frame_table(df_te, ob_te)
rows = [{'điểm': c, 'AUROC dấu dương': roc_auc_score(va.y, smooth(va, va[c])), 'AUROC dấu âm': roc_auc_score(va.y, smooth(va, -va[c]))}
        for c in COMPS + ['total', 'nobj', 'pix', 'bg']]
log('AUROC val (đã làm mượt σ=1):'); print(pd.DataFrame(rows).round(4).to_string(index=False))

AUROC trong từng video (mức khung, không bị thiên lệch giữa các video) - thước đo đúng cho "phát hiện vật thể bất thường":

In [ ]:
for c in ['total', 'H4', 'H5', 'H1']:
    per = [roc_auc_score(va.y[va.video == v], va[c][va.video == v]) for v in va.video.unique() if 0 < va.y[va.video == v].mean() < 1]
    log(f'{c}: AUROC trung bình trong từng video {np.mean(per):.3f} ·', ' '.join(f'{x:.2f}' for x in per))

## 5. Biến thể để nộp (chọn bằng LB public)

In [ ]:
# Ghép bằng rank-average với điểm starter (khác nền trung vị, đổi dấu). Trên val dùng -bg, tương đương starter.
rank = lambda x: pd.Series(np.asarray(x)).rank(pct=True).values
ST_FILE = 'submission_starter.csv'
st_te = (pd.read_csv(ST_FILE).set_index('id')['score'].loc[te.id].values if os.path.exists(ST_FILE) else -te.bg.values)
st_va = -va.bg.values
VARIANTS = {
    'obj_H4':                lambda d, st: smooth(d, d.H4),
    'obj_H3':                lambda d, st: smooth(d, d.H3),
    'obj_total':             lambda d, st: smooth(d, d.total),
    'combo_starter_2H4':     lambda d, st: rank(st) + 2 * rank(smooth(d, d.H4)),
    'combo_starter_H3':      lambda d, st: rank(st) + rank(smooth(d, d.H3)),
    'combo_starter_2H4_H3':  lambda d, st: rank(st) + 2 * rank(smooth(d, d.H4)) + rank(smooth(d, d.H3)),
}
res = []
for name, f in VARIANTS.items():
    pd.DataFrame({'id': te.id, 'score': f(te, st_te)}).to_csv(f'submission_v2_{name}.csv', index=False)
    s = f(va, st_va); res.append({'biến thể': f'v2_{name}', 'val AUROC': round(roc_auc_score(va.y, s), 4),
                                  'val AP': round(average_precision_score(va.y, s), 4), 'LB public': ''})
res = pd.DataFrame(res).sort_values('val AUROC', ascending=False)
sub = pd.DataFrame({'id': te.id, 'score': VARIANTS['combo_starter_2H4'](te, st_te)})   # cách ghép tốt nhất trên LB
assert len(sub) == len(pd.read_csv(f'{DATA}/sample_submission.csv')) and np.isfinite(sub.score).all()
sub.to_csv('submission.csv', index=False)
log(f'đã ghi submission.csv (= v2_combo_starter_2H4) và {len(VARIANTS)} file submission_v2_<biến thể>.csv')
print(res.to_string(index=False))